In [21]:
import numpy as np
import pandas as pd
import requests

In [22]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "/home/renku/work/input-data/Project 1 - DTB/bioreaction_template.csv"
SUBMISSION_PATH = "submissions/blend_team.csv"   # predictions to submit (same rows as the template)
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [23]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 7,1,1,2026-11-21T18:06:56.327299,6.745,6.649,7.234,7.446,7.018
1,Team 15,2,4,2026-11-21T20:41:02.740184,6.627,6.513,7.748,7.658,7.137
2,Team 04,3,9,2026-11-21T16:16:43.084759,7.151,6.903,7.546,6.969,7.142
3,Team 23,4,11,2026-11-21T23:01:18.628816,7.280,7.216,6.822,8.149,7.367
4,Team 10,5,87,2026-11-21T23:05:45.720608,7.183,7.178,8.297,8.241,7.725
5,Team 01,6,1,2026-11-21T17:04:14.729206,6.975,6.802,8.615,8.551,7.736
6,Team 18,7,8,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
7,TEMPLATE,8,43,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517


In [24]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
df = pd.read_csv(SUBMISSION_PATH, dtype={"batch_id": str})
# Sanity checks: same rows / order as the template, no missing or negative values
assert (df[["batch_id", "elapsed_time"]].values == template[["batch_id", "elapsed_time"]].values).all()
assert df[TARGETS].notna().all().all() and (df[TARGETS] > 0).all().all()
metrics = submit(df[["batch_id", "elapsed_time", *TARGETS]], user="Team 10")
metrics

{'vcd_mape': 7.183,
 'tcd_mape': 7.178,
 'substrate_mape': 8.297,
 'byproduct_mape': 8.241,
 'mean_mape': 7.725}

In [25]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 7,1,1,2026-11-21T18:06:56.327299,6.745,6.649,7.234,7.446,7.018
1,Team 15,2,4,2026-11-21T20:41:02.740184,6.627,6.513,7.748,7.658,7.137
2,Team 04,3,9,2026-11-21T16:16:43.084759,7.151,6.903,7.546,6.969,7.142
3,Team 23,4,11,2026-11-21T23:01:18.628816,7.280,7.216,6.822,8.149,7.367
4,Team 10,5,88,2026-11-21T23:05:45.720608,7.183,7.178,8.297,8.241,7.725
5,Team 01,6,1,2026-11-21T17:04:14.729206,6.975,6.802,8.615,8.551,7.736
6,Team 18,7,8,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
7,TEMPLATE,8,43,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517
